# DINOv3 Image Embedding, Deduplication & LanceDB Workflow

This notebook demonstrates the full DINOv3-based embedding pipeline used for
panda-wire deduplication in the SAM3 fine-tuning project.

What it covers:
1. Load a local DINOv3 checkpoint.
2. Load image records from a COCO annotation file.
3. Extract L2-normalized CLS-token embeddings.
4. Deduplicate embeddings by cosine similarity.
5. Store unique embeddings in a LanceDB table.
6. Query the table for similar images.
7. Append new images to an existing table while deduplicating against it.
8. Build a clean COCO dataset from the unique records.

The helper module is `sam3_fine_tune.preprocessing.dinov3_dedup`.

In [ ]:
#| default_exp preprocessing.dinov3_dedup_notebook

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

## 1. Configuration

Set paths for the COCO dataset, the DINOv3 checkpoint, and the LanceDB directory.
Adjust these to point to your own data.

In [ ]:
from pathlib import Path

# COCO dataset to embed
coco_annotation = Path("/home/hpc_storage/data/projects/userB/sam3_data/9cat_separated_wires_v3/train/annotation.json")
image_root = Path("/home/hpc_storage/data/projects/userB/sam3_data/9cat_separated_wires_v3/train/images")

# Local DINOv3 checkpoint
model_checkpoint = Path("/home/hpc_storage/Models/dinov3_models/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")

# Where to store the LanceDB table
db_uri = Path("/home/hpc_storage/data/projects/userB/sam3_data/dinov3_dedup_db")
table_name = "panda_wire_embeddings"

# Categories we want to embed (panda-wire categories)
categories = ["thick_wire_panda", "thin_wire_panda"]
split = "train"

# Deduplication threshold (cosine similarity)
threshold = 0.999

print("coco_annotation:", coco_annotation)
print("exists:", coco_annotation.exists())
print("image_root exists:", image_root.exists())
print("model_checkpoint exists:", model_checkpoint.exists())

## 2. Import the DINOv3 helper module

If the module is not on the current branch, you can check it out from
`feat/dinov3-panda-wire-dedup` or copy `sam3_fine_tune/preprocessing/dinov3_dedup.py`
into your working tree.

In [ ]:
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:%(name)s:%(message)s")

from sam3_fine_tune.preprocessing.dinov3_dedup import (
    Record,
    build_transform,
    load_dinov3_model,
    load_coco_panda_records,
    extract_embeddings,
    deduplicate_embeddings,
    create_embedding_table,
    query_similar,
    append_unique_records,
    build_unique_panda_dataset,
)

print("Imported dinov3_dedup helpers.")

## 3. Load image records from COCO

We keep one record per image that belongs to the requested categories.
The category label comes from the first matching annotation for that image.

In [ ]:
records = load_coco_panda_records(
    coco_annotation,
    image_root,
    categories=categories,
    split=split,
)
print(f"Loaded {len(records)} records")
print("First 3 records:")
for r in records[:3]:
    print(f"  {r.id} | {r.category} | {r.image_path}")

## 4. Load DINOv3 model

The helper clones the official `facebookresearch/dinov3` repository on first use
(if a local clone is not provided), loads the state dict, and returns the model
in eval mode.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

transform = build_transform(img_size=518)
model = load_dinov3_model(
    model_checkpoint,
    model_name="vit_large",
    img_size=518,
    device=device,
)

## 5. Extract embeddings

Run the images through DINOv3 and L2-normalize the CLS-token outputs.
`num_workers=0` is useful inside a notebook to avoid multiprocessing issues.

In [ ]:
# For a quick test, limit the number of records
max_images = None  # e.g. 100 for a quick smoke test
records_subset = records[:max_images] if max_images else records

embeddings, records_out = extract_embeddings(
    model,
    transform,
    records_subset,
    batch_size=32,
    num_workers=0,
    device=device,
)

print("embeddings shape:", embeddings.shape)
print("embeddings dtype:", embeddings.dtype)
print("L2 norm check (first 5):", (embeddings[:5] ** 2).sum(axis=1))

## 6. Deduplicate embeddings

Two images are considered duplicates if their cosine similarity is above the
threshold. The function returns the indices to keep and a mapping from removed
indices to their representative index.

In [ ]:
keep_indices, duplicate_map = deduplicate_embeddings(
    embeddings,
    threshold=threshold,
    chunk_size=1024,
    device=device,
)

print(f"Kept {len(keep_indices)} / {len(embeddings)} images")
print(f"Removed {len(duplicate_map)} duplicates")
print("First 5 duplicate mappings:", list(duplicate_map.items())[:5])

## 7. Create a LanceDB table

Store the unique records and their embeddings in a LanceDB table for fast
similarity search.

In [ ]:
unique_records = [records_out[i] for i in keep_indices]
unique_embeddings = embeddings[keep_indices]

table = create_embedding_table(
    db_uri,
    table_name,
    unique_records,
    unique_embeddings,
)

print(f"Table '{table_name}' created at {db_uri}")
print(f"Rows: {len(unique_records)}")

## 8. Query similar images

Pick a query image, embed it, and ask LanceDB for the most similar records.

In [ ]:
from PIL import Image
import numpy as np

query_image_path = unique_records[0].image_path
print("Query image:", query_image_path)

with Image.open(query_image_path).convert("RGB") as img:
    tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    query_emb = model(tensor).detach().cpu().numpy()

norm = np.linalg.norm(query_emb, axis=1, keepdims=True)
query_emb = query_emb / np.where(norm == 0, 1.0, norm)
query_emb = query_emb.astype(np.float32)

results = query_similar(
    db_uri,
    table_name,
    query_emb[0],
    top_k=5,
    metric="cosine",
)

results

## 9. Append new images to an existing table

When a new batch of images arrives, extract embeddings and append only the
records that are not near-duplicates of items already stored in the table.

In [ ]:
# Example: use the same annotation as the "new" batch (in practice this is a
# different dataset, e.g. newly collected panda-wire photos).
new_records = load_coco_panda_records(
    coco_annotation,
    image_root,
    categories=categories,
    split="new_batch",
)

new_embeddings, _ = extract_embeddings(
    model,
    transform,
    new_records[:200] if len(new_records) > 200 else new_records,
    batch_size=32,
    num_workers=0,
    device=device,
)

table, appended = append_unique_records(
    db_uri,
    table_name,
    new_records[:200] if len(new_records) > 200 else new_records,
    new_embeddings,
    threshold=threshold,
    device=device,
)

print(f"Appended {appended} new unique records")

## 10. Build a deduplicated COCO dataset

Use the duplicate map from step 6 to write a new `annotation.json` (and optional
copy of images) containing only unique panda-wire images plus all other
(non-panda) categories unchanged.

In [ ]:
import json

duplicate_map_path = db_uri / f"{table_name}_duplicate_map.json"
duplicate_map_path.write_text(
    json.dumps({str(k): int(v) for k, v in duplicate_map.items()}), encoding="utf-8"
)

output_dir = Path("/home/hpc_storage/data/projects/userB/sam3_data/9cat_separated_wires_v3_unique_panda_train")

summary = build_unique_panda_dataset(
    annotation_path=coco_annotation,
    image_root=image_root,
    duplicate_map_path=duplicate_map_path,
    output_dir=output_dir,
    categories=categories,
    split=split,
    copy_images=True,
    num_workers=4,
)

for k, v in summary.items():
    print(f"{k}: {v}")

## 11. Optional: CLI equivalents

The same operations are available as command-line scripts, which is how they
are usually submitted to the HPC CPU queue:

```bash
# Build embedding table
python -m sam3_fine_tune.preprocessing.dinov3_dedup build \
  --annotation /path/to/train/annotation.json \
  --image-root /path/to/train/images \
  --db-uri /path/to/db \
  --model-checkpoint /path/to/dinov3_vitl16_pretrain_*.pth \
  --verbose

# Query similar images
python -m sam3_fine_tune.preprocessing.dinov3_dedup query \
  --query-image /path/to/query.png \
  --db-uri /path/to/db \
  --model-checkpoint /path/to/dinov3_vitl16_pretrain_*.pth \
  --top-k 5

# Append new images
python -m sam3_fine_tune.preprocessing.dinov3_dedup add \
  --annotation /path/to/new/annotation.json \
  --image-root /path/to/new/images \
  --db-uri /path/to/db \
  --model-checkpoint /path/to/dinov3_vitl16_pretrain_*.pth

# Build unique dataset
python -m sam3_fine_tune.preprocessing.dinov3_dedup build-dataset \
  --annotation /path/to/train/annotation.json \
  --image-root /path/to/train/images \
  --duplicate-map /path/to/db/panda_wire_embeddings_duplicate_map.json \
  --output-dir /path/to/output
```

In [ ]:
#| hide
from nbdev.showdoc import *